# 09b — PRIO-GRID Feature Engineering Methods

**Depends on:** notebook 09 (PRIO-GRID pull), notebook 04 (UCDP-GED events)  
**Reads from ADLS:** `raw/prio_grid/`, `raw/ucdp_ged/`  
**Writes to ADLS:** `raw/prio_grid/{RUN_DATE}/priogrid_engineered_features.parquet`  

## Purpose

Notebook 09 produces a simple country-year aggregation (mean/sum/max per variable).
This notebook builds seven additional feature families from the same grid-cell data
that are more predictive of instability outcomes because they capture *within-country
variation* — the distributional shape, spatial concentration, and temporal shocks
that flat country-level averages erase.

All outputs are at **country-year** resolution and are ready to join into the
feature matrix built in notebook 14.

## Seven methods

| # | Method | Instability mechanism |
|---|---|---|
| 1 | **Distributional aggregates** | Dispersion of NTL/pop captures urban–rural fractures |
| 2 | **NTL Gini / darkness fraction** | Within-country economic inequality proxy |
| 3 | **Population-weighted climate stress** | Which populations face rainfall/temp volatility |
| 4 | **Conflict cell density (UCDP-GED join)** | Spatial breadth of active violence |
| 5 | **Lootable resource population exposure** | Grievance potential of resource enclaves |
| 6 | **Economic polarization (core vs. periphery)** | Primate city concentration, regional neglect |
| 7 | **NTL economic shock index** | Sudden economic deterioration signal |

## Required environment variables
```
ADLS_ACCOUNT_NAME
ADLS_CONTAINER  (default: 'data')
```

In [ ]:
import os
import re
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

from azure.identity import DefaultAzureCredential
import adlfs

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 40)

## Configuration

In [ ]:
ADLS_ACCOUNT_NAME = os.environ["ADLS_ACCOUNT_NAME"]
ADLS_CONTAINER    = os.getenv("ADLS_CONTAINER", "data")
RUN_DATE          = datetime.utcnow().strftime("%Y%m%d")

PANEL_START_YEAR = 2000
PANEL_END_YEAR   = 2024

# NTL thresholds (DMSP-OLS calibrated scale: 0–63)
NTL_DARK_THRESHOLD   = 3    # below this = essentially unlit
NTL_URBAN_THRESHOLD  = 20   # above this = urban / semi-urban

# Climate stress: top-quartile rainfall SD within country-year is "high stress"
CLIMATE_STRESS_QUANTILE = 0.75

# NTL shock: flag if pop-weighted NTL drops more than this many SDs below 5-yr baseline
NTL_SHOCK_THRESHOLD = 1.5

print(f"Run date     : {RUN_DATE}")
print(f"Panel years  : {PANEL_START_YEAR}–{PANEL_END_YEAR}")

## ADLS helpers

In [ ]:
credential = DefaultAzureCredential()
storage_options = {
    "account_name": ADLS_ACCOUNT_NAME,
    "credential":   credential,
}

def adls_path(subpath: str) -> str:
    return (
        f"abfss://{ADLS_CONTAINER}@{ADLS_ACCOUNT_NAME}"
        f".dfs.core.windows.net/{subpath}"
    )

def write_parquet(df: pd.DataFrame, subpath: str) -> None:
    path = adls_path(subpath)
    df.to_parquet(path, storage_options=storage_options, index=False, engine="pyarrow")
    print(f"  Written {len(df):,} rows → {path}")

def read_latest_parquet(prefix: str, filename_hint: str = "") -> pd.DataFrame | None:
    """Read the parquet from the lexicographically latest date partition under prefix."""
    fs = adlfs.AzureBlobFileSystem(
        account_name=ADLS_ACCOUNT_NAME, credential=credential
    )
    full_prefix = f"{ADLS_CONTAINER}/{prefix}"
    try:
        entries = fs.ls(full_prefix, detail=False)
    except FileNotFoundError:
        print(f"  WARNING: {full_prefix} not found")
        return None

    date_dirs = sorted(
        [e for e in entries if re.search(r'/\d{8}(/|$)', e)],
        reverse=True,
    )
    if not date_dirs:
        print(f"  WARNING: no date partitions under {full_prefix}")
        return None

    pattern = f"{date_dirs[0]}/*{filename_hint}*.parquet" if filename_hint else f"{date_dirs[0]}/*.parquet"
    files = [
        f"abfss://{ADLS_CONTAINER}@{ADLS_ACCOUNT_NAME}.dfs.core.windows.net/"
        + f.replace(f"{ADLS_CONTAINER}/", "", 1)
        for f in fs.glob(pattern)
    ]
    if not files:
        print(f"  WARNING: no parquet files at {date_dirs[0]}")
        return None

    dfs = [pd.read_parquet(p, storage_options=storage_options) for p in files]
    df  = pd.concat(dfs, ignore_index=True) if len(dfs) > 1 else dfs[0]
    print(f"  Loaded {len(df):,} rows ← {date_dirs[0]}")
    return df

## Load PRIO-GRID data

We need two tables produced by notebook 09:
- **Static** — one row per grid cell with geographic invariants (`bdist1`, `mountains_mean`, etc.)
- **Yearly** — one row per (cell, year) with time-varying variables (`pop_gpw_sum`, `nlights_calib_mean`, etc.)

And one table from notebook 04:
- **UCDP-GED events** — one row per conflict event with a `priogrid_gid` field, used in Method 4.

In [ ]:
# ── PRIO-GRID static ──────────────────────────────────────────────────────────
df_static = read_latest_parquet("raw/prio_grid", filename_hint="static")

# ── PRIO-GRID yearly ─────────────────────────────────────────────────────────
df_yearly = read_latest_parquet("raw/prio_grid", filename_hint="yearly")

# ── UCDP-GED events (for Method 4) ───────────────────────────────────────────
df_ged = read_latest_parquet("raw/ucdp_ged", filename_hint="events")

for name, df in [("static", df_static), ("yearly", df_yearly), ("ged", df_ged)]:
    if df is not None:
        df.columns = [c.lower().strip() for c in df.columns]
        print(f"  {name:8s}: {df.shape}  cols={list(df.columns[:8])}...")
    else:
        print(f"  {name:8s}: NOT LOADED — some methods will be skipped")

## Country crosswalk (GW code → ISO3)

PRIO-GRID uses Gleditsch-Ward (`gwno`) country codes. We map these to ISO3 for joining
to the country-year feature matrix.

In [ ]:
_cw_path = next(
    (p for p in [Path("../data/country_crosswalk.csv"), Path("data/country_crosswalk.csv")]
     if p.exists()),
    None,
)
if _cw_path:
    df_cw = pd.read_csv(_cw_path, dtype=str)
    df_cw["gw_numeric"] = pd.to_numeric(df_cw["gw_numeric"], errors="coerce")
    gw_to_iso3 = dict(zip(df_cw["gw_numeric"], df_cw["iso3"]))
    print(f"Crosswalk loaded: {len(df_cw)} countries")
else:
    print("WARNING: country_crosswalk.csv not found — gwno→iso3 mapping unavailable")
    gw_to_iso3 = {}

## Merge static variables into yearly panel

Joins geographic invariants (border distance, terrain, forest cover) onto the yearly
time-varying rows so that every cell-year has both geographic and temporal attributes.
Also maps `gwno` → `iso3` and filters to the panel years.

In [ ]:
df_panel = None

if df_yearly is not None:
    df_panel = df_yearly.copy()

    # Merge static geography onto yearly rows
    if df_static is not None:
        static_cols = [c for c in df_static.columns
                       if c not in df_panel.columns or c == "gid"]
        df_panel = df_panel.merge(df_static[static_cols], on="gid", how="left")

    # Resolve country code: prefer gwno column
    gwno_col = next((c for c in df_panel.columns if c in ("gwno", "gwnoa", "gwno_a")), None)
    if gwno_col and gw_to_iso3:
        df_panel["iso3"] = pd.to_numeric(df_panel[gwno_col], errors="coerce").map(gw_to_iso3)

    # Ensure numeric types
    df_panel["year"] = pd.to_numeric(df_panel["year"], errors="coerce").astype("Int64")
    for col in ["pop_gpw_sum", "nlights_calib_mean", "rainfall_sd", "temp_sd", "ttime_mean"]:
        if col in df_panel.columns:
            df_panel[col] = pd.to_numeric(df_panel[col], errors="coerce")

    # Filter to panel years
    df_panel = df_panel[
        df_panel["year"].between(PANEL_START_YEAR, PANEL_END_YEAR)
    ].copy()

    print(f"Cell-year panel : {len(df_panel):,} rows")
    print(f"Years           : {df_panel['year'].min()}–{df_panel['year'].max()}")
    print(f"Cells           : {df_panel['gid'].nunique():,}")
    print(f"Countries (iso3): {df_panel['iso3'].nunique() if 'iso3' in df_panel.columns else 'n/a'}")
else:
    print("WARNING: yearly panel not loaded — cannot proceed with feature engineering")

## Method 1 — Enhanced distributional aggregates

**What it adds:** Standard notebook 09 aggregates NTL and population to country-year means,
sums, and max values. Those statistics describe the *level* but miss the *shape* of the
distribution within each country. A country where 5% of cells hold 90% of the light is
structurally different from one with evenly distributed light — even if their means are
identical.

**Features produced:**

| Feature | Description |
|---|---|
| `prio_ntl_mean` | Mean nighttime light intensity across cells |
| `prio_ntl_std` | Std dev of NTL — within-country economic dispersion |
| `prio_ntl_p10` | 10th-percentile NTL (how dark are the dark areas?) |
| `prio_ntl_p90` | 90th-percentile NTL (how bright are the bright areas?) |
| `prio_ntl_p90p10_ratio` | p90 / p10 — distributional spread (high = stark inequality) |
| `prio_pop_sum` | Total country population (sum of cell populations) |
| `prio_pop_std` | Std dev of cell-level population — spatial concentration of people |
| `prio_pop_top10cell_share` | Fraction of population in the 10% most populated cells |

**Why it matters for instability models:**
High `prio_ntl_p90p10_ratio` and `prio_pop_top10cell_share` both proxy for the
spatial concentration of wealth and people that Cederman et al. (2011) link to
horizontal inequality and conflict risk.

In [ ]:
df_m1 = pd.DataFrame()

if df_panel is not None and "iso3" in df_panel.columns:
    ntl_col = "nlights_calib_mean"
    pop_col = "pop_gpw_sum"

    grp = df_panel.groupby(["iso3", "year"])

    agg = pd.DataFrame()

    if ntl_col in df_panel.columns:
        ntl = grp[ntl_col]
        agg["prio_ntl_mean"]        = ntl.mean()
        agg["prio_ntl_std"]         = ntl.std()
        agg["prio_ntl_p10"]         = ntl.quantile(0.10)
        agg["prio_ntl_p90"]         = ntl.quantile(0.90)
        p10 = agg["prio_ntl_p10"].replace(0, np.nan)
        agg["prio_ntl_p90p10_ratio"] = agg["prio_ntl_p90"] / p10

    if pop_col in df_panel.columns:
        pop = grp[pop_col]
        agg["prio_pop_sum"] = pop.sum()
        agg["prio_pop_std"] = pop.std()

        # Fraction of population in the top-10% most populated cells
        def _top10_share(s):
            s = s.dropna()
            if s.sum() == 0 or len(s) == 0:
                return np.nan
            threshold = s.quantile(0.90)
            return s[s >= threshold].sum() / s.sum()

        agg["prio_pop_top10cell_share"] = grp[pop_col].apply(_top10_share)

    df_m1 = agg.reset_index()
    print(f"Method 1: {len(df_m1):,} rows × {len(df_m1.columns)} columns")
    print(df_m1.describe().round(3))
else:
    print("Method 1 skipped — panel not available")

## Method 2 — NTL Gini coefficient and darkness fraction

**What it adds:** The Gini coefficient of nighttime light values across a country's grid
cells is a direct spatial proxy for economic inequality within a country. A Gini of 0
means every cell is equally lit; a Gini near 1 means nearly all light is concentrated
in a handful of cells. This is distinct from aggregate GDP inequality measures because
it captures the geographic dimension — whether inequality is expressed as spatial
separation between rich and poor regions.

The darkness fraction captures a complementary dimension: what share of the country's
area is in effective darkness, regardless of the distribution among lit areas.

**Features produced:**

| Feature | Description |
|---|---|
| `prio_ntl_gini` | Gini coefficient of NTL across cells (0 = equal, 1 = concentrated) |
| `prio_ntl_darkness_frac` | Fraction of cells with NTL < 3 (essentially unlit) |
| `prio_ntl_urban_frac` | Fraction of cells with NTL > 20 (urban / semi-urban proxy) |

**Why it matters:** Høyland et al. (2012) and Lessmann & Seidel (2017) show that
within-country spatial inequality (proxied by NTL Gini) is a robust predictor of
separatist conflict and ethnic civil war onset, over and above national-level
inequality measures like the Gini of household income.

In [ ]:
def _gini(values: np.ndarray) -> float:
    """Gini coefficient for a 1-D array of non-negative values."""
    v = values[~np.isnan(values)]
    v = v[v >= 0]
    if len(v) == 0 or v.sum() == 0:
        return np.nan
    v = np.sort(v)
    n = len(v)
    idx = np.arange(1, n + 1)
    return float((2 * (idx * v).sum()) / (n * v.sum()) - (n + 1) / n)


df_m2 = pd.DataFrame()

if df_panel is not None and "iso3" in df_panel.columns:
    ntl_col = "nlights_calib_mean"

    if ntl_col not in df_panel.columns:
        print(f"Method 2 skipped — '{ntl_col}' not in panel")
    else:
        grp = df_panel.groupby(["iso3", "year"])[ntl_col]

        ntl_gini = grp.apply(lambda s: _gini(s.values)).rename("prio_ntl_gini")

        ntl_darkness = grp.apply(
            lambda s: (s < NTL_DARK_THRESHOLD).sum() / max(len(s), 1)
        ).rename("prio_ntl_darkness_frac")

        ntl_urban = grp.apply(
            lambda s: (s > NTL_URBAN_THRESHOLD).sum() / max(len(s), 1)
        ).rename("prio_ntl_urban_frac")

        df_m2 = pd.concat([ntl_gini, ntl_darkness, ntl_urban], axis=1).reset_index()

        print(f"Method 2: {len(df_m2):,} rows × {len(df_m2.columns)} columns")
        print(df_m2[["prio_ntl_gini", "prio_ntl_darkness_frac", "prio_ntl_urban_frac"]].describe().round(3))
else:
    print("Method 2 skipped — panel not available")

## Method 3 — Population-weighted climate stress

**What it adds:** Simple country-year means of rainfall variability (`rainfall_sd`) and
temperature variability (`temp_sd`) treat all cells equally. But a drought that hits
an empty desert is not the same as one that hits a densely populated agricultural
region. Population-weighting shifts the measure from geographic exposure to *human*
exposure — the fraction of the population actually experiencing climate volatility.

**Features produced:**

| Feature | Description |
|---|---|
| `prio_pop_wtd_rainfall_sd` | Population-weighted mean rainfall SD across cells |
| `prio_pop_wtd_temp_sd` | Population-weighted mean temperature SD across cells |
| `prio_high_stress_pop_frac` | Fraction of population in cells above the country's 75th-percentile rainfall SD |

**Why it matters:** Burke et al. (2015) and Hsiang et al. (2013) establish a causal
link between climate shocks and conflict, but the effect is concentrated where people
live. `prio_high_stress_pop_frac` directly captures what share of the population is
exposed to the worst climate volatility in their country — a more precise grievance
signal than a national average.

In [ ]:
def _pop_weighted_mean(df_grp, value_col, weight_col):
    """Population-weighted mean of value_col within each (iso3, year) group."""
    def _pwm(g):
        v = g[value_col].values.astype(float)
        w = g[weight_col].values.astype(float)
        mask = ~(np.isnan(v) | np.isnan(w)) & (w > 0)
        if mask.sum() == 0:
            return np.nan
        return np.average(v[mask], weights=w[mask])
    return df_grp.apply(_pwm)


df_m3 = pd.DataFrame()

if df_panel is not None and "iso3" in df_panel.columns:
    pop_col = "pop_gpw_sum"
    rain_col = "rainfall_sd"
    temp_col = "temp_sd"

    missing = [c for c in [pop_col, rain_col] if c not in df_panel.columns]
    if missing:
        print(f"Method 3 skipped — missing columns: {missing}")
    else:
        grp = df_panel.groupby(["iso3", "year"])

        rows = {}

        if rain_col in df_panel.columns:
            rows["prio_pop_wtd_rainfall_sd"] = _pop_weighted_mean(grp, rain_col, pop_col)

        if temp_col in df_panel.columns:
            rows["prio_pop_wtd_temp_sd"] = _pop_weighted_mean(grp, temp_col, pop_col)

        # Fraction of population in high-stress cells (above country 75th-pct rainfall SD)
        def _high_stress_pop_frac(g):
            if rain_col not in g.columns or pop_col not in g.columns:
                return np.nan
            r = g[rain_col].values.astype(float)
            p = g[pop_col].fillna(0).values.astype(float)
            if p.sum() == 0 or np.isnan(r).all():
                return np.nan
            threshold = np.nanquantile(r, CLIMATE_STRESS_QUANTILE)
            return p[r >= threshold].sum() / p.sum()

        rows["prio_high_stress_pop_frac"] = grp.apply(_high_stress_pop_frac)

        df_m3 = pd.DataFrame(rows).reset_index()
        print(f"Method 3: {len(df_m3):,} rows × {len(df_m3.columns)} columns")
        print(df_m3.drop(columns=["iso3", "year"]).describe().round(3))
else:
    print("Method 3 skipped — panel not available")

## Method 4 — Conflict cell density (UCDP-GED × PRIO-GRID join)

**What it adds:** UCDP-GED event records each include a `priogrid_gid` field — the
0.5° cell where the event occurred. Joining events back to the grid lets us compute
the *spatial footprint* of conflict within each country-year: how many cells had any
violence, what fraction of the country's area that represents, and what fraction of
the population lived inside conflict-affected cells.

This distinguishes spatially concentrated conflicts (insurgent enclaves, border wars)
from diffuse ones (nationwide breakdown), which have different escalation dynamics and
different policy implications.

**Features produced:**

| Feature | Description |
|---|---|
| `prio_conflict_cell_count` | Number of distinct cells with ≥1 state-based violence event |
| `prio_conflict_cell_frac` | Conflict cells / total country cells — spatial footprint |
| `prio_conflict_pop_frac` | Population in conflict cells / total country population |
| `prio_conflict_deaths_per_cell` | Battle deaths (best estimate) per active conflict cell |

**Why it matters:** `prio_conflict_cell_frac` distinguishes a country where violence
is confined to one border region from one in nationwide collapse. `prio_conflict_pop_frac`
captures how many civilians are in the conflict zone — a direct input to humanitarian
crisis escalation models.

In [ ]:
df_m4 = pd.DataFrame()

if df_panel is not None and df_ged is not None and "iso3" in df_panel.columns:
    # State-based violence only (type_of_violence == 1)
    ged_sb = df_ged[df_ged.get("type_of_violence", pd.Series(dtype=int)) == 1].copy() \
        if "type_of_violence" in df_ged.columns \
        else df_ged.copy()

    gid_col  = next((c for c in ged_sb.columns if c in ("priogrid_gid", "priogrid_id", "gid")), None)
    yr_col   = next((c for c in ged_sb.columns if c == "year"), None)
    best_col = next((c for c in ged_sb.columns if c == "best"), None)

    if not gid_col or not yr_col:
        print(f"Method 4 skipped — UCDP-GED missing gid/year columns (found: {list(ged_sb.columns[:10])})")
    else:
        ged_sb[yr_col]  = pd.to_numeric(ged_sb[yr_col],  errors="coerce").astype("Int64")
        ged_sb[gid_col] = pd.to_numeric(ged_sb[gid_col], errors="coerce")

        # Aggregate GED to cell-year: distinct event count + total deaths
        ged_agg = (
            ged_sb.groupby([gid_col, yr_col], as_index=False)
            .agg(
                event_count=(gid_col, "count"),
                **({best_col: (best_col, "sum")} if best_col else {}),
            )
            .rename(columns={gid_col: "gid", yr_col: "year"})
        )

        # Join cell-level population and iso3 from panel (one row per cell-year)
        cell_info = df_panel[["gid", "year", "iso3", "pop_gpw_sum"]].drop_duplicates()
        ged_cells = ged_agg.merge(cell_info, on=["gid", "year"], how="left")

        # Total cells per country-year (denominator)
        total_cells = (
            cell_info.groupby(["iso3", "year"])["gid"]
            .nunique()
            .rename("total_cells")
            .reset_index()
        )

        # Total population per country-year
        total_pop = (
            cell_info.groupby(["iso3", "year"])["pop_gpw_sum"]
            .sum()
            .rename("total_pop")
            .reset_index()
        )

        # Conflict-cell aggregates per country-year
        ged_cy = (
            ged_cells.groupby(["iso3", "year"], as_index=False)
            .agg(
                prio_conflict_cell_count=("gid", "nunique"),
                prio_conflict_pop=("pop_gpw_sum", "sum"),
                **({best_col: (best_col, "sum")} if best_col in ged_cells.columns else {}),
            )
        )

        df_m4 = (
            ged_cy
            .merge(total_cells, on=["iso3", "year"], how="left")
            .merge(total_pop,   on=["iso3", "year"], how="left")
        )

        df_m4["prio_conflict_cell_frac"] = (
            df_m4["prio_conflict_cell_count"] / df_m4["total_cells"].replace(0, np.nan)
        )
        df_m4["prio_conflict_pop_frac"] = (
            df_m4["prio_conflict_pop"] / df_m4["total_pop"].replace(0, np.nan)
        )
        if best_col in df_m4.columns:
            df_m4["prio_conflict_deaths_per_cell"] = (
                df_m4[best_col] / df_m4["prio_conflict_cell_count"].replace(0, np.nan)
            )
            df_m4 = df_m4.drop(columns=[best_col])

        df_m4 = df_m4.drop(columns=["prio_conflict_pop", "total_cells", "total_pop"])

        print(f"Method 4: {len(df_m4):,} rows × {len(df_m4.columns)} columns")
        print(df_m4.drop(columns=["iso3", "year"]).describe().round(3))
else:
    print("Method 4 skipped — panel or UCDP-GED events not available")